In [1]:
import pandas as pd
import numpy as np
import hashlib
import os

# Pandas display options for better visibility
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

In [2]:
RAW_DATA_PATH = '../data/raw/hotel_bookings_course_release_v1.csv'
HASH_FILE_PATH = '../data/SHA256SUMS.txt'

def verify_dataset_hash(csv_path, hash_path):
    """Calculates the SHA-256 hash of the CSV and compares it to the provided text file."""
    if not os.path.exists(csv_path):
        return "Dataset not found. Please ensure it is placed in the data/raw/ folder."
    if not os.path.exists(hash_path):
        return "Hash file not found. Please ensure SHA256SUMS.txt is in the data/ folder."
    
    # Calculate SHA-256 of the CSV
    sha256_hash = hashlib.sha256()
    with open(csv_path, "rb") as f:
        for byte_block in iter(lambda: f.read(4096), b""):
            sha256_hash.update(byte_block)
    calculated_hash = sha256_hash.hexdigest()
    
    # Read the expected hash from the text file
    with open(hash_path, "r") as f:
        expected_hash_line = f.readline().strip()
        expected_hash = expected_hash_line.split()[0]
        
    print(f"Calculated Hash: {calculated_hash}")
    print(f"Expected Hash:   {expected_hash}\n")
    
    if calculated_hash == expected_hash:
        return "- Integrity Check Passed: The dataset matches the course release exactly."
    else:
        return "- Integrity Check Failed: The hashes do not match."

print(verify_dataset_hash(RAW_DATA_PATH, HASH_FILE_PATH))

Calculated Hash: 7c2ae42a7353905ea136e5c2287f17c92c5435826598bfbb8491c6f0c7b1fc06
Expected Hash:   7c2ae42a7353905ea136e5c2287f17c92c5435826598bfbb8491c6f0c7b1fc06

- Integrity Check Passed: The dataset matches the course release exactly.


In [3]:
df = pd.read_csv(RAW_DATA_PATH)

print(f"Dataset Shape: {df.shape[0]:,} rows (bookings) and {df.shape[1]} columns (features)")
display(df.head(3))

Dataset Shape: 119,390 rows (bookings) and 32 columns (features)


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,country,market_segment,distribution_channel,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,reserved_room_type,assigned_room_type,booking_changes,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,0.0,0,BB,PRT,Direct,Direct,0,0,0,C,C,3,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,0.0,0,BB,PRT,Direct,Direct,0,0,0,C,C,4,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,0.0,0,BB,GBR,Direct,Direct,0,0,0,A,C,0,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02


In [4]:
# Isolation of categorical columns
cat_cols = df.select_dtypes(exclude=[np.number]).columns

# Calculate missingness
cat_missing = df[cat_cols].isnull().sum()
cat_missing_pct = (cat_missing / len(df)) * 100

# Format as a clean DataFrame
cat_missing_df = pd.DataFrame({
    'Missing Values': cat_missing,
    'Percentage (%)': cat_missing_pct
}).sort_values(by='Percentage (%)', ascending=False)

print("Missingness Summary: Categorical Variables")

# Display only columns that actually have missing values
display(cat_missing_df[cat_missing_df['Missing Values'] > 0])

Missingness Summary: Categorical Variables


,Missing Values,Percentage (%)
country,488,0.408744


In [5]:
# Core numerical variables for our baseline (and ADR for profiling/checks)
core_num_vars = [
    'lead_time', 
    'stays_in_weekend_nights', 
    'stays_in_week_nights', 
    'adults', 
    'children', 
    'babies',
    'previous_cancellations',
    'previous_bookings_not_canceled',
    'adr'
]

print("Outlier & Heavy Tail Check: Core Numerical Variables")
display(df[core_num_vars].describe().round(2))

# Quick check for the specific extreme values the professor warned us about
print("\nSpecific Anomaly Checks:")
print(f"Bookings with ADR < 0: {(df['adr'] < 0).sum()}")
print(f"Bookings with ADR > 1000: {(df['adr'] > 1000).sum()}")
print(f"Bookings with 0 adults: {(df['adults'] == 0).sum()}")

Outlier & Heavy Tail Check: Core Numerical Variables


,lead_time,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,previous_cancellations,previous_bookings_not_canceled,adr
count,119390.00,119390.00,119390.00,119390.00,119386.0,119390.00,119390.00,119390.00,119390.00
mean,104.01,0.93,2.50,1.86,0.1,0.01,0.09,0.14,101.83
std,106.86,1.00,1.91,0.58,0.4,0.10,0.84,1.50,50.54
min,0.00,0.00,0.00,0.00,0.0,0.00,0.00,0.00,-6.38
25%,18.00,0.00,1.00,2.00,0.0,0.00,0.00,0.00,69.29
50%,69.00,1.00,2.00,2.00,0.0,0.00,0.00,0.00,94.58
75%,160.00,2.00,3.00,2.00,0.0,0.00,0.00,0.00,126.00
max,737.00,19.00,50.00,55.00,10.0,10.00,26.00,72.00,5400.00



Specific Anomaly Checks:
Bookings with ADR < 0: 1
Bookings with ADR > 1000: 1
Bookings with 0 adults: 403
